# Deployment Performance Report

*Generated by DoML — Do Machine Learning*

In [ ]:
import os
import random
import json
import glob
import time
import subprocess
import warnings
import numpy as np
import pandas as pd
import psutil
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Markdown, display

# REPR-01: random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# REPR-02: project root from env var
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/home/jovyan/work'))
WEB_SERVICE_URL = os.environ.get('DOML_WEB_SERVICE_URL', 'http://localhost:8080')

warnings.filterwarnings('ignore')

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'WEB_SERVICE_URL: {WEB_SERVICE_URL}')

In [ ]:
# Load deployment_metadata.json
meta_files = sorted(glob.glob(str(PROJECT_ROOT / 'src' / '*' / '*' / 'deployment_metadata.json')))
if not meta_files:
    raise FileNotFoundError(
        'No deployment_metadata.json found under src/. '
        'Run a deployment target first (deploy-cli / deploy-web / deploy-wasm).'
    )

meta_path = meta_files[-1]
with open(meta_path) as f:
    meta = json.load(f)

target = meta['target']
model_file = meta['model_file']
feature_schema = meta['feature_schema']
problem_type = meta.get('problem_type', 'regression')
model_name = meta.get('model_name', 'model')
version = meta.get('version', 'v1')

model_path = PROJECT_ROOT / model_file
deploy_dir = Path(meta_path).parent

# Reconstruct category_map from feature_schema
category_map = {
    entry['name']: entry['categories']
    for entry in feature_schema
    if entry.get('categories') is not None
}

print(f'Metadata: {meta_path}')
print(f'Target: {target}')
print(f'Model: {model_name} ({model_file})')
print(f'Problem type: {problem_type}')
print(f'Version: {version}')
print(f'Deploy dir: {deploy_dir}')
print(f'Category map features: {list(category_map.keys())}')

In [ ]:
# Load test data
processed_files = sorted(glob.glob(str(PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_*.csv')))
if not processed_files:
    raise FileNotFoundError(
        'No preprocessed_*.csv found in data/processed/. '
        'Run the EDA phase (/doml-data-understanding) first.'
    )

data_path = processed_files[-1]
df_all = pd.read_csv(data_path)
test_df = df_all.tail(100).reset_index(drop=True)

# Select columns available in feature_schema
schema_cols = [entry['name'] for entry in feature_schema]
available_cols = [c for c in schema_cols if c in test_df.columns]
test_df = test_df[available_cols]

print(f'Data source: {data_path}')
print(f'Test shape: {test_df.shape}')
display(test_df.head(3))

In [ ]:
# Load in-memory model + capture load memory delta (PERF-05)
process = psutil.Process()
rss_before_load = process.memory_info().rss
pipeline = joblib.load(model_path)
rss_after_load = process.memory_info().rss
model_load_memory_mb = (rss_after_load - rss_before_load) / (1024 ** 2)

print(f'Model loaded: {model_path}')
print(f'Model load memory delta: {model_load_memory_mb:.2f} MB')

In [ ]:
# Target routing — define run_single(row) and run_batch(df)

if target == 'cli_binary':
    binary_path = deploy_dir / 'dist' / 'predict' / 'predict'
    if not binary_path.exists():
        raise FileNotFoundError(
            f'CLI binary not found: {binary_path}. '
            'Run /doml-deploy-cli first.'
        )

    def run_single(row):
        result = subprocess.run(
            [str(binary_path), '--input', json.dumps(row)],
            capture_output=True, text=True, check=True
        )
        return json.loads(result.stdout.strip())

    def run_batch(df):
        return [run_single(r.to_dict()) for _, r in df.iterrows()]

elif target == 'web_service':
    import requests

    try:
        health_resp = requests.get(WEB_SERVICE_URL + '/health', timeout=5)
        health_resp.raise_for_status()
    except Exception as exc:
        raise RuntimeError(
            f'Web service health check failed at {WEB_SERVICE_URL}/health. '
            'The deploy-model.md Step 12 workflow should have started the container before '
            'executing this notebook. Ensure the service is running and DOML_WEB_SERVICE_URL '
            f'is set correctly. Original error: {exc}'
        ) from exc

    def run_single(row):
        return requests.post(WEB_SERVICE_URL + '/predict', json=row, timeout=30).json()

    def run_batch(df):
        return [run_single(r.to_dict()) for _, r in df.iterrows()]

elif target == 'onnx_wasm':
    import onnxruntime as _ort

    onnx_path = deploy_dir / 'model.onnx'
    if not onnx_path.exists():
        raise FileNotFoundError(
            f'ONNX model not found: {onnx_path}. '
            'Run /doml-deploy-wasm first.'
        )

    _sess = _ort.InferenceSession(str(onnx_path))
    _input_name = _sess.get_inputs()[0].name

    def _encode_row(row):
        encoded = []
        for entry in feature_schema:
            name = entry['name']
            val = row.get(name)
            if name in category_map:
                cats = category_map[name]
                encoded.append(float(cats.index(val)) if val in cats else 0.0)
            else:
                encoded.append(float(val) if val is not None else 0.0)
        return np.array([encoded], dtype=np.float32)

    def run_single(row):
        return _sess.run(None, {_input_name: _encode_row(row)})[0].tolist()

    def run_batch(df):
        return [run_single(r.to_dict()) for _, r in df.iterrows()]

    print('ONNX target: benchmarked via Python onnxruntime (approximation). Browser performance may vary.')

else:
    raise ValueError(f'Unknown target: {target}')

print(f'Target routing configured: {target}')

In [ ]:
# Single-prediction latency benchmark (PERF-02)
WARMUP_RUNS = 5
TIMED_RUNS = 1000

sample_row = test_df.iloc[0].to_dict()

# Warm-up (not timed)
for _ in range(WARMUP_RUNS):
    run_single(sample_row)

# Timed runs
latencies_ms = []
for _ in range(TIMED_RUNS):
    t0 = time.perf_counter()
    run_single(sample_row)
    t1 = time.perf_counter()
    latencies_ms.append((t1 - t0) * 1000)

mean_latency_ms = np.mean(latencies_ms)
std_latency_ms = np.std(latencies_ms)
p50 = np.percentile(latencies_ms, 50)
p95 = np.percentile(latencies_ms, 95)
p99 = np.percentile(latencies_ms, 99)

print(f'Single-prediction latency over {TIMED_RUNS} runs (after {WARMUP_RUNS} warm-up):')
print(f'  Mean: {mean_latency_ms:.3f} ms ± {std_latency_ms:.3f} ms')
print(f'  p50:  {p50:.3f} ms')
print(f'  p95:  {p95:.3f} ms')
print(f'  p99:  {p99:.3f} ms')

# Plot latency distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(latencies_ms, bins=50, color='steelblue', edgecolor='none')
ax.axvline(mean_latency_ms, color='red', linestyle='--', label=f'Mean {mean_latency_ms:.2f} ms')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('Single-Prediction Latency Distribution')
ax.legend()
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'notebooks' / 'latency_distribution.png'), dpi=120)
plt.close()
print('Latency histogram saved to notebooks/latency_distribution.png')

In [ ]:
# Batch latency benchmark (PERF-03)
BATCH_SIZES = [10, 100, 1000, 10_000]

batch_results = []
for bs in BATCH_SIZES:
    # Repeat test_df rows to fill the batch size
    repeats = (bs // len(test_df)) + 1
    batch_df = pd.concat([test_df] * repeats, ignore_index=True).iloc[:bs]

    t0 = time.perf_counter()
    run_batch(batch_df)
    t1 = time.perf_counter()

    total_ms = (t1 - t0) * 1000
    per_item_ms = total_ms / bs
    batch_results.append({
        'batch_size': bs,
        'total_ms': round(total_ms, 2),
        'per_item_ms': round(per_item_ms, 4)
    })
    print(f'Batch {bs:>6}: total={total_ms:.1f} ms, per-item={per_item_ms:.4f} ms')

batch_df_results = pd.DataFrame(batch_results)
display(batch_df_results)

In [ ]:
# Parity test — HARD AssertionError (PERF-04)
in_memory_preds = pipeline.predict(test_df)
deployed_raw = run_batch(test_df)

def _flatten(raw):
    result = []
    for item in raw:
        if isinstance(item, dict):
            val = item.get('prediction', item.get('result', next(iter(item.values()))))
        elif isinstance(item, list) and len(item) == 1:
            val = item[0]
        else:
            val = item
        result.append(val)
    return np.array(result)

deployed_preds = _flatten(deployed_raw)

if problem_type == 'regression':
    max_diff = np.max(np.abs(in_memory_preds.astype(float) - deployed_preds.astype(float)))
    assert np.allclose(in_memory_preds.astype(float), deployed_preds.astype(float), atol=1e-4), (
        f'PARITY FAILURE (regression): max abs diff = {max_diff:.6f} exceeds atol=1e-4. '
        'Deployed model predictions do not match in-memory model.'
    )
    print(f'Parity PASSED (regression): max abs diff = {max_diff:.8f}')

elif problem_type in ('classification', 'binary_classification', 'multiclass_classification'):
    assert np.array_equal(in_memory_preds.astype(str), deployed_preds.astype(str)), (
        'PARITY FAILURE (classification): deployed predictions do not match in-memory model labels.'
    )
    print(f'Parity PASSED (classification): all {len(in_memory_preds)} labels match')

elif problem_type == 'clustering':
    assert np.array_equal(in_memory_preds.astype(str), deployed_preds.astype(str)), (
        'PARITY FAILURE (clustering): deployed predictions do not match in-memory cluster IDs.'
    )
    print(f'Parity PASSED (clustering): all {len(in_memory_preds)} cluster IDs match')

else:
    print(f'Parity test skipped for problem_type={problem_type}')

In [ ]:
# Per-prediction memory delta (PERF-05)
MEMORY_RUNS = 10

memory_deltas_mb = []
for _ in range(MEMORY_RUNS):
    rss_before = process.memory_info().rss
    run_single(sample_row)
    rss_after = process.memory_info().rss
    delta_mb = (rss_after - rss_before) / (1024 ** 2)
    memory_deltas_mb.append(delta_mb)

mean_pred_memory_mb = np.mean(memory_deltas_mb)

print(f'Model load memory delta: {model_load_memory_mb:.2f} MB')
print(f'Per-prediction memory delta (mean of {MEMORY_RUNS} runs): {mean_pred_memory_mb:.4f} MB')

In [ ]:
# Throughput projection (PERF-06)
throughput_rps = 1000.0 / mean_latency_ms

print(f'Projected throughput (single-threaded): {throughput_rps:.1f} req/s')
print('Note: actual throughput under concurrency depends on worker count and I/O overlap.')

In [ ]:
# Summary table
summary = {
    'Metric': [
        'Target',
        'Model',
        'Model load memory (MB)',
        'Mean latency (ms)',
        'Std latency (ms)',
        'p95 latency (ms)',
        'Per-prediction memory (MB)',
        'Projected throughput (req/s)',
        'Parity result'
    ],
    'Value': [
        target,
        model_name,
        f'{model_load_memory_mb:.2f}',
        f'{mean_latency_ms:.3f}',
        f'{std_latency_ms:.3f}',
        f'{p95:.3f}',
        f'{mean_pred_memory_mb:.4f}',
        f'{throughput_rps:.1f}',
        'PASSED'
    ]
}

summary_df = pd.DataFrame(summary)
display(summary_df.style.hide(axis='index'))

print('\nBatch latency results:')
print(batch_df_results.to_string(index=False))

## Performance Summary

*This section will be replaced by a Claude-generated narrative by the deploy-model.md Step 12 workflow.*